![hslu_logo.png](./img/hslu_logo.png)


<hr style="border:1px solid black">

<h1 style="text-align:center;font-size:50px"><b>AAI - FS25</b></h1>
<p style="text-align:center;font-size:40px">Week 04</p>

---

# Transfer Learning

---
---



# Table of contents for week 04
1. [Introduction to Transfer Learning](#intro_trans)
2. [DataSet and DataLoader of PyTorch](#data_loader)
     1. [DataSet and DataLoader](#data_sets)
     2. [Data Augmentation](#data_augmentation)
3. [Transfer Learning on food and celebrity data](#transfer)


## Introduction to Transfer Learning <a name="intro_trans"></a>
In the previous chapters we have seen the power of neural networks in particular of CNNs for the classification of images in different categories. The strength of the CNNs lies in the combination of the features extraction part - via the successive convolutional and pooling layers - with the final MLP-stage for the classification (<a href="#fig1">Fig.1</a>). But even the very simply CNN architecture studied in [sw03.03.cnn.ipynb](http://localhost:8888/notebooks/Kursunterlagen/SW03/sw03.03.cnn.ipynb#) has more than 300'000 parameters, mainly due to the first fully connected layer at the beginning of the classifier stage. In order to train such a large number of parameters a sufficiently large number of training samples is required that cover the typical inter- and intra-class variances of the data set that the classifier is intended for. For the present work we used the (fashion)MNIST and CIFAR10 image set comprising 70'000 and 60'000 images respectively. But in practice of often do not have access to such large sample sets and only a few dozen or hundred sample images are available. It is nevertheless possible to train deep neural networks by making use of the so-called *transfer learning* technique.

State-of-the-art CNNs as e.g. vgg16 (<a href="#fig1">Fig.1</a>) as studied in [sw03.ipynb](http://localhost:8888/notebooks/Kursunterlagen/SW03/sw03.ipynb#) were trained on image sets comprising hundreds of thousands or even millions of images (e.g. [ImageNet](https://en.wikipedia.org/wiki/ImageNet#)). The feature extraction part of these CNNs should have learned very generic (low- up to medium-level) features that should be suited for any kind of visual object <a id="anker1" href="#ref1">[1]</a>. 

<br>
<img src="img/vgg16_segm.png" alt="Drawing" width="800" />
<a id="fig1">Fig.1:</a> Architecture of the VGG16 CNN <a id="anker2" href="#ref2">[2]</a>.
<br>

---

The idea of transfer learning is to use make use of these existing architectures and to adapt them to the problem one wants to solve. Different strategies exists, which are summarised under the label *transfer learning*.

- Retraining the final layers of the original architecture:<br>
  Building up on the idea of <a href="#fig1">Fig.1</a> that the feature extraction part is generic and the information on the specific class is mainly contained in the classifier part a frequent transfer learning scheme consists in fixing all but the last layer of an existing architecture and then train the network with the own dataset. As a rule of thumb the number of layers that are not *freezed* (i.e. trainable) increases with the number of training samples available. Thus it is not unfrequent to train also a few of the last features extraction layers i.e., the higher level object features in particalar in combination with *image augmentation* techniques that allow to increase the amount fo training data.

<br>
<img src="img/vgg16_transfer.png" alt="Drawing" width="800" />
<a id="fig2">Fig.2:</a> Transfer learning through retrainig a few of the final layers of an existing VGG16 CNN.</a>
<br>

---

- Replacing and retraining the classifier:<br>
  The original VGG16 architecture was trained on 1000 different categories (so-called [ImageNet-1K](https://en.wikipedia.org/wiki/ImageNet#ImageNet-1K)), which explains the fully connected layer with 1000 neurons at the output in <a href="#fig1">Fig.1</a>. If the classification task we intend to use the VGG16 for is simpler i.e., has much less categories a frequent scheme is to replace the entire classifier with simplified version as e.g. shown in <a href="#fig3">Fig.3</a>. This is the architecture, which we will use later for the transfer learning tasks studied. There are two advantages.
  - Because the feature extraction part does not require a given input size we can work with other (e.g. smaller) image sizes. Below we will use an input of 3x128x128. After the final pooling layer the activation maps of a given input image will have a size of 512x4x4, which is a total of 8'192 features. This compares to 25'088 features of the original architecture.
  - Furthermore we add a hidden layer of 256 neurons "only" as compared to 4096 in VGG16. Thus the total number of weights in the first fully connected layer is 2’097’152 as compared to 102’760’448 and we obtain a considerable reduction (~50) of the total number of parameters which will be of advantage both in the training and the inference process. Nevertheless the representational capacity of the model will turn out to be sufficent to model the few (5 or 8) categories of our sample problem.<br>
*Remark:* In <a href="#fig3">Fig.3</a> the training is shown to be only over the layers of the new classifier but it is not uncommon to *unfreeze* also the last layers of the features extraction part and we will adopt this strategy below.

<br>
<img src="img/vgg16_transfer_1.png" alt="Drawing" width="800" />
<a id="fig3">Fig.3:</a> Transfer learning through replacing and retrainig the classificaton part of an existing VGG16 CNN.</a>
<br>

---

- *Fine Tuning*<br>
  This is in principle a special version of the first transfer learning strategy but the expression is frequently used when the entire original architecture is used and trained (all layers unfreezed) but at a very low learning rate. This strategy is used in particular if the original model contains already the desired class(es) and only a particular 'view' of the class(es) is required. E.g. the original model contains the class 'person' and we want to distinguish sitting and standing persons<br>
  The default choice of a small learning rate is in fact a very important point to remember for all transfer learning schemes because even networks pretrained on hundreds of thousands images "forget" rather quickly the original information when retrained with new data at a high learning rate. This is refered to a [catastrophic inference](https://en.wikipedia.org/wiki/Catastrophic_interference#).

So far we used datasets available easily through the ML-frameworks, which in our case is `torchvision.datasets`. For the transfer learning tasks we will the first time work with own data. To read and "feed" this data efficiently to the neural networks the ML-frameworks provide so-called dataloaders. We will introduce the dataloader of `PyTorch` in the next chapter and also discuss the important topic of *data augmentation* because it is implicitly handled by the data loaders.

<a id="ref1" href="#anker1">[1]</a> This is similar to the "training" of the human brain during the first years after birth, which later allows us to "detect" also unknown objects when we see them for the first time.

<a id="ref2" href="#anker2">[2]</a> Simonyan, Karen, and Andrew Zisserman. "Very deep convolutional networks for large-scale image recognition." arXiv preprint arXiv:1409.1556 (2014).

---


## DataSet and DataLoader of PyTorch <a name="data_loader"></a>

The [DataLoader](https://pytorch.org/docs/stable/data.html#torch.utils.data.DataLoader#) class  provides a generic way to prepare and sample datasets for the training step. In the notation of the code snipets used so far it will take over the role of the class MiniBatches and the data normalisation. It will require as input a [Dataset](https://pytorch.org/vision/stable/datasets.html#) class, which is mainly a container for the data itself. PyTorch provides already a large choice of build-in datasets  but one can also define custom – i.e. own – datasets. The (Fashion)MNIST and CIFAR10 datasets we used so far are part of the available PyTorch models and are of the type Dataset. The Dataset and DataLoader concepts allow to keep the code for maintaining the data set and the model well separated (see also this [link](https://pytorch.org/tutorials/beginner/basics/data_tutorial.html#) for a concise introduction to these concepts). For the actual training process a further tool is used which is the transformation and – in a later step – the augmentation of the data. These are actually part of the torchvision sub-library of the PyTorch project  (as are the available datasets) and are passed through the transform argument to the Dataset class. torchvision provides a large set of available [transformations](https://pytorch.org/vision/stable/transforms.html#start-here#) and we will make use of them below. 

### DataSet and DataLoader <a name="data_sets"></a>

We will study the concepts of the PyTorch DataSet and DataLoader in the following iPython notebook. We will proceed in two steps. First we will use the (fashion)MNIST as an example of an predefined DataSet and in a second step we will define our own DataSet.

**Exercise:** 
**[sw04.01.torch_data-set-loader.ipynb](http://localhost:8888/notebooks/Kursunterlagen/SW04/sw04.01.torch_data-set-loader.ipynb)**

- Cell [2]<br>
  Here the transform concept is introduced. We have to put it at the top because it is required for the definition of the dataset in the cell underneath. Our data transformation is named `my_transform` a parameter to be given to the dataset definition. It is a composition of three individual transformations, which are:
   - `ToImage`: Conversion to a PyTorch tensor from type PIL image
   - `ToDtype`: Tpye conversion from uint8 to float32
   - `Normalize`: Normalisation of data (here Min-Max Normalisation)

<img src="img/transforms.png" alt="Drawing" width="800" />

- Cell [3]<br>
  This statement was so far "hidden" in the file `utils.py` but we show it here explicitly because we want to illustrate that the inbuild imagesets are of type DataSet and that image transformations can be applied on the fly when accessing the sets. 

<img src="img/mnist_set.png" alt="Drawing" width="600" />  

- Cell [4]<br>
  Here we check that the predefined PyTorch datasets are of the type Dataset. To be precise they realise the interface of the class Dataset. As we will see below, this requires – apart from the `__init__` method, which is obvious – the methods `__len__` and `__getitem__` to be implemented. 

<img src="img/dataset_if.png" alt="Drawing" width="700" />  

- Cell [5]<br>
  Here we just illustrate that the training_data instance has the transform method as attribute:

<img src="img/transform_if.png" alt="Drawing" width="450" />  

- Cell [6] and [7]<br>
  These two cells now introduce the actual DataLoader concept. The constructor of the class `DataLoader` receives as input the instance of the `Dataset` class `training_data` we prepared beforehand. In addition, further training parameters as the batch size and sampling strategy (random shuffling or not) can be chosen. Refer to the PyTorch [documentation](https://pytorch.org/docs/stable/data.html#torch.utils.data.DataLoader#) for a complete list of arguments. <br>
Once the DataLoader is set up an iterator is defined and used to get the data batch-wise, including the corresponding labels. 
When executing the cell note the size of the returned tensor, which is: `torch.Size([256, 1, 28, 28])`<br>

<img src="img/dataloader.png" alt="Drawing" width="800" />  

- Cell [8]<br>
  A few images are ploted as tile-image. Because the min-max-normalisation is performed (c.f. Cell [9]) we rescale the images because the plot function requires the range between $[0,1]$.
  
- Cell [9]<br>
  Here we verify the application of the transformations during the iteration step over the data. It is important to notice that the transformations are done on the fly by the DataLoader when iterating over the data and are not performed on the original data, as you can see when executing the cell [8].

<img src="img/transform_mnist.png" alt="Drawing" width="800" />  

- Cell [10]<br>
  Here a loop over the entire dataset is performed. We also measure the time for this operation.

<img src="img/full_iteration.png" alt="Drawing" width="700" />    

It is now straight forward to extend this concept to own data. This only required to "wrap" the data into a costum DataSet. This is illustrated in the following part of the iPython notebook.

- Cell [11]<br>
  Here we prepare the use of our own data. It is organised in subfolders of a `root_folder`, each subfolder corresponding to a category. Two sets exist, which are the `train` and the `test` set. Below the folder structure for `test` is shown (`train` identical).<br>
  The function `create_file_frame` (in `utils.py`) iterates through these folders and returns a list of pandas dataframes (mainly a two-column list) with the image paths and corresponding labels (as integer values) for both `train`and `test` sets. In addition `csv`-files in the `root_folder` are created with the same information.

<img src="img/file_tree.png" alt="Drawing" width="250" />   

- Cell [12]<br>
  This is the defintion of our own DataSet. It inherits from the original class `DataSet` and has to fulfill the corresponding interface with the methods `__len__` and `__getitem__`. In addition you can notice that the image transformation is only applied in the `__getitem__` call i.e., not to the original image. This is relevant for the data augmentation process that we will discuss below.

<img src="img/dataset_own.png" alt="Drawing" width="800" />   

- Cell [13]<br>
  Here we define a new transformation for the "food"-data, which we will use below for the transfer learning task. We center crop the image to square size (from 300x400 to 300x300) and scale them to the final size 128x128.

<img src="img/transforms_2.png" alt="Drawing" width="700" />   

- Cell [14] and [15]<br>
  These cells are a "repetition" of the Cells [6] till [8] but now with our own data. This illustrates how the encapsulation of the own data into the `DataSet` class allows the generic access to the DataLoader functionality.

As we have illustrated in Cells [9] and [12] the transformation is only applied in the `__getitem__` call. The reason is that the transformations covers the possibility for a wide range of image transformations, which can be used to increase the training data set. We will discuss this in the section below.
  
### Data Augmentation <a name="data_augmentation"></a>

The size of the training data is always an issue when training deep neural network architectures with many parameters. For image-based problems efficient augmentation schemes exist that reproduce certain changes also present in real world data. The following 
<a href="#fig4">Fig.4</a>) shows some examples of typical image augmentation schemes (image taken from [Link](https://www.bishopbook.com#)):

- a) original image
- b) horizontal flip
- c) scaling
- d) translation
- e) rotation
- f) brightness change
- g) additive noise
- h) color shift

<br>
<img src="img/augmentation_cat.png" alt="Drawing" width="800" />
<a id="fig4">Fig.4:</a> Different image augmentation schemes applied. Details see text.</a>
<br>

---

The idea is to apply augmentation schemes according to the desired invariances of the classifier. As the image changes in <a href="#fig4">Fig.4</a> are also frequent in natural images their use during training makes certainly sens.

With the concept of the the DataSet and DataLoader developed above we now have an efficient possibility to apply these augmentation schemes on the fly during the training process. 

**Exercise:** 
**[sw04.01.torch_data-set-loader.ipynb](http://localhost:8888/notebooks/Kursunterlagen/SW04/sw04.01.torch_data-set-loader.ipynb)**

We return to the previous iPython notebook and continue with Cell [16].

- Cell [16]<br>
  Here we use again the class `torchvision.transforms` ([Link](https://pytorch.org/vision/main/transforms.html#)) but now for image augmentation as illustrated in <a href="#fig4">Fig.4</a>. As an example we apply a random rotation in the intervall $[-25°,25°]$, a random scaling in the interval $[0.75,1.25]$, a random shear in the interval $[-15°,15°]$ and a random horizontal flip with a probability of 0.5 (later the transformations will be less pronouced).

<img src="img/augmentation_own.png" alt="Drawing" width="800" />  

- Cell [17] and [18]<br>
  These now reproduce the setup of the DataLoader now with the above transform and in Cell [18] the corresponding result for each batch of 16 augmented images can be visualised. Each execution will recover a new batch of images with (randomly chosen) different augmentations.

<img src="img/augmentation_food.png" alt="Drawing" width="600" />    

## Transfer Learning on food and celebrity data <a name="transfer"></a>

We will now apply the concepts discussed above on a two examples. The first is a typical example form the area of process-automation with the classification of five types of fruit (`food`). It is of average compexity. The second one is the attempt to classify eight different celibrities from a limited number of images downloaded randomly from the internet and is of fairly high complexity.

**Exercise:** 
**[sw04.02.transfer_learn.ipynb](http://localhost:8888/notebooks/Kursunterlagen/SW04/sw04.02.transfer_learn.ipynb)**

We discuss step by step the different cell inputs and results.

- Cell [2]<br>
  Here we prepare the part of the VGG16 model that we want to use for the transfer learning.
  - The parameter `vgg_param_file` is the file name where we store the weigths of the VGG16            features extraction part.
  - We import the `vgg16` model and weights from the `torchvision.models` package.
  - We setup the `vgg16` model with default weights. You may analyse the architecture the full `vgg16` model (with `print()`), which consists of two main parts the `features` and the `classifier`. We only want to make use of the feature-part and replace the classifier by our own layers (<a href="#fig3">Fig.3</a>).
  - Therefore we store the `features` of the `vgg16` model in an own variable `model_to_save`, which serves only to store the model to file via the call to `torch.save`.

<img src="img/save_vgg_features.png" alt="Drawing" width="800" /> 

- Cell [3]<br>
  Here you can choose either the `food` dataset (we introduced already in the notebook [sw04.01.torch_data-set-loader.ipynb](http://localhost:8888/notebooks/Kursunterlagen/SW04/sw04.01.torch_data-set-loader.ipynb) or the more challenging `celebrities` set (`problem_type = 0` or `1`). Because the image sizes between the two sets are not compatible we also define a default size (e.g. `default_size=(300, 400)` for food), which is at the first step of the transform stage in Cell [4].

- Cell [4]<br>
  We define a set of transforms, one for the training (`train_transform`) including the data augmentation (`v2.RandomAffine`) and one for the validation (`val_transform`) without augmentation. We apply moderate affine tansformations (e.g. scaling from $[0.95,1.05]$) corresponding to changes expected in real images. Note the dataset specific resizing at the beginning of the tansforms: `v2.Resize(default_size, antialias=True)`

- Cell [5] - [7]<br>
  These cells are identical to Cells [6] till [8] from the notebook [sw04.01.torch_data-set-loader.ipynb](http://localhost:8888/notebooks/Kursunterlagen/SW04/sw04.01.torch_data-set-loader.ipynb) and set up a DataLoader based on an own DataSet and illustrate the transform opration.

- Cell [8]<br>
  This function will setup our transfer learning model.
  - It will load the vgg16 classifier stored in the file `param_file`.
  - It will set all but the last two convolutional layers to non-trainable.
  - Then a classifier consisting of one fully connected and an output layer is added. These layers are by default trainable. Note the `Dropout` layer, which serves as a regularisation to avoid overfitting.
  - Note the flag `print_freeze`, which allows to print the status of all layers with respect to the trainable feature.
  
<img src="img/setup_model.png" alt="Drawing" width="800" /> 

- Cell [9]<br>
  This serves only to investigate our transfer learning model e.g. to visualise (using the flag `print_freeze`) which layers are trainable.

- Cell [10] and [11]<br>
  These define the `NeuralNetwork` class (Cell [10]) and initiate the training procedure (Cell [11]). Several major difference with respect to our previous implementations occured:
  - The constructor of the `NeuralNetwork` class simply recieves our model (from Cell [8]) as input.
  - The data is wrapped into our own DataSet class (`MyDataset`) and the torch DataLoader is used for the training (c.f. `NeuralNetwork.optimise`).
  - Because the neural network is larger and the inference slower we determine the train cost and error during the trainig loop in `NeuralNetwork.optimise` and only determine the validation error and cost after the entire epoch. This is the reason why the training error at the beginning might be larger than the validation error, because the former is an average over the entire epoch the latter is only evaluated on the final - optimised - version of the network.
  - Note that we train only a few epochs with rather small learning rate.
  - You will notice a very fast learing procedure with a close to zero training and validation error.
 
<img src="img/transfer_training.png" alt="Drawing" width="500" /> 
 
- Cell [12] and [13]<br>
  The test error on the independet set will be around 5% (Cell [12]). You can plot the wrongly classified images using Cell [13]. You will notice that some of the confusions between apples and peaches are even for a human observer difficult to resolve.

<img src="img/transfer_false.png" alt="Drawing" width="800" /> 

- Cell [14]<br>
  Here you can visualise the full confusion matrix, where the apple <-> peach confusion is again the most prominent.

<img src="img/transfer_confusion.png" alt="Drawing" width="550" /> 

The results show the effectiveness of the transfer learning approach. You may now move up to Cell [3] and choose the celebrities dataset. The following <a href="#fig5">Fig.5</a> shows some of the 
560 images (480 train and 80 test). The original images have been download randomly from the internet using a key-word search with a subsequent manual filtering to rule out false or inappropriate samples. 

<br>
<img src="img/celebrities.png" alt="Drawing" width="800" />
<a id="fig5">Fig.5:</a> Some samples of the celebrities data set.</a>
<br>

---

You can perform the training and should end up with a test error rate of some 25% - 30%. This seems to be fairly bad but you should recall that random guessing of 8 classes implies an error rate of 77.5 %. In addition the confusion matrix reveals, that e.g. man and women are quite well distinguished and the overall confusion performance is - considering the complexity of the task even for humans - is not bad.

<img src="img/transfer_confusion_celebrities.png" alt="Drawing" width="550" /> 


**Exercise:** 
**[sw04.03.cnn_food.ipynb](http://localhost:8888/notebooks/Kursunterlagen/SW04/sw04.03.cnn_food.ipynb)**

As a comparison we train a cnn from scratch using the given iPython notebook. It should be easy to understand the code because the basis is the notebook from the previous week on the configurable CNN-architecture ([sw03.03.cnn.ipynb](http://localhost:8888/notebooks/Kursunterlagen/SW03/sw03.03.cnn.ipynb)) with the extension of the DataSet and DataLoader introduced above. 

- Cell [8]<br>
  Here you can start the training procedure and the CNN-architecture is printed:

<img src="img/cnn_scratch_model.png" alt="Drawing" width="800" /> 

- The training will converge, but with quite large fluctuations despite the same learning rate as used for the transfer learning.

<img src="img/cnn_scratch_training.png" alt="Drawing" width="500" />


- Cell [9] to [11]<br>
  The test error rate will be typically around 12% - 15% with confusion matrix as shown below:

<img src="img/cnn_scratch_confusion.png" alt="Drawing" width="550" /> 

Thus when compared to the transfer learning approach a training from scratch is apparently considerably less successful.